AdaBoost Practice Project: Loan Default Prediction
Intermediate ML Project
 Classification


I recommend practising AdaBoost using a Bank Loan Default Prediction System. This is an industry-relevant problem because banks and financial institutions use classification models to identify customers who are likely to default on their loans.
1. Problem statement
A bank provides personal loans to customers. However, some customers fail to repay their loans on time, resulting in financial losses for the bank.

The bank wants to develop a Machine Learning model using AdaBoost Classifier that predicts whether a customer is likely to default on a loan based on their financial and personal information.

Your task is to build a classification model that learns from historical customer data and predicts the default status of new customers.
2. Expected output
The model should predict one of two classes:
Prediction	Meaning
0	Customer will not default
1	Customer is likely to default

Example:

Input customer:
- Age: 28
- Annual income: 450000
- Loan amount: 300000
- Credit score: 580
- Previous defaults: 1
Expected prediction:

In [19]:
import pandas as pd


In [20]:
df=pd.read_csv('ada_boost_loan_default_practice.csv')
cdf=df

In [21]:
cdf.head()

,Age,Annual_Income,Loan_Amount,Credit_Score,Employment_Years,Debt_To_Income,Previous_Defaults,Credit_Utilization,Loan_Purpose,Home_Ownership,Employment_Type,Defaulted
0,25,528538,516187,624,8.5,0.259,1,0.010,Home Improvement,Rent,Self-employed,1
1,59,505237,1012396,692,5.7,0.298,0,0.262,Personal,Own,Salaried,0
2,53,992179,940197,697,4.2,0.156,0,0.604,Personal,Mortgage,Self-employed,0
3,42,1019920,452269,585,2.6,0.170,0,0.183,Home Improvement,Rent,Self-employed,0
4,42,484285,98530,615,3.2,0.369,1,0.487,Personal,Mortgage,Salaried,1


In [22]:
cdf.Home_Ownership.value_counts()
cdf['Home_Ownership']=cdf.Home_Ownership.replace({
    "Rent":1,
    "Mortgage":2,
    "Own":3
})

C:\Users\Pawan\AppData\Local\Temp\ipykernel_16792\2826195357.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cdf['Home_Ownership']=cdf.Home_Ownership.replace({


In [23]:
cdf['Employment_Type']=cdf.Employment_Type.replace({
    "Salaried":1,
    "Self-employed":2,
    "Contract":3
})

C:\Users\Pawan\AppData\Local\Temp\ipykernel_16792\2422559837.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cdf['Employment_Type']=cdf.Employment_Type.replace({


In [24]:

X=cdf.drop(['Defaulted','Loan_Purpose'],axis=1)
Y=cdf['Defaulted']


BASELINE MODEL 

In [25]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
x_train,x_test,y_train,y_test=train_test_split(X,Y,random_state=42,test_size=0.20,stratify=Y)


base_tree=DecisionTreeClassifier(max_depth=1)
base_tree.fit(x_train,y_train)
y_pred=base_tree.predict(x_test)
print('Accuracy :',accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print('Classification Report :')
print(classification_report(y_test,y_pred))

Accuracy : 0.67
[[488 174]
 [156 182]]
Classification Report :
              precision    recall  f1-score   support

           0       0.76      0.74      0.75       662
           1       0.51      0.54      0.52       338

    accuracy                           0.67      1000
   macro avg       0.63      0.64      0.64      1000
weighted avg       0.67      0.67      0.67      1000



In [32]:

from sklearn.ensemble import AdaBoostClassifier

ada=AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50,
    random_state=42,
    learning_rate=1,
)

ada.fit(x_train,y_train)

ada_y_pred=ada.predict(x_test)

print("Accuracy_score:",accuracy_score(y_test,ada_y_pred))

print(classification_report(y_test,ada_y_pred))

Accuracy_score: 0.69
              precision    recall  f1-score   support

           0       0.73      0.84      0.78       662
           1       0.56      0.39      0.46       338

    accuracy                           0.69      1000
   macro avg       0.64      0.62      0.62      1000
weighted avg       0.67      0.69      0.67      1000



In [ ]:
print("Best accuracy  :",ada.estimator_)

stump_details = pd.DataFrame({
    'Round': range(1, len(ada.estimators_) + 1),
    'Weighted error': ada.estimator_errors_[:len(ada.estimators_)],
    'Voting weight': ada.estimator_weights_[:len(ada.estimators_)]
})
display(stump_details.head(10).round(4))

Best Accuracy : DecisionTreeClassifier(max_depth=1)


,Round,Weighted error,Voting weight
0,1,0.3287,0.7138
1,2,0.3593,0.5786
2,3,0.4195,0.3250
3,4,0.4389,0.2458
4,5,0.4684,0.1265
5,6,0.4432,0.2282
6,7,0.4747,0.1012
7,8,0.4416,0.2346
8,9,0.4766,0.0937
9,10,0.4610,0.1562


In [42]:
from sklearn.model_selection import cross_val_score

cv=cross_val_score(ada,x_train,y_train,scoring='accuracy',cv=5)

print("Accuracies :",cv.round(4))
print("Mean CV Accuracy :",cv.mean())

Accuracies : [0.705  0.7038 0.675  0.71   0.735 ]
Mean CV Accuracy : 0.70575


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'learning_rate': [0.01, 0.1, 0.5, 1.0],
    'estimator__max_depth': [1, 2, 3]
}


grid=GridSearchCV(
    estimator=ada,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)

In [45]:
grid.fit(x_train,y_train)

,estimator,AdaBoostClass...ndom_state=42)
,param_grid,"{'estimator__max_depth': [1, 2, ...], 'learning_rate': [0.01, 0.1, ...], 'n_estimators': [50, 100, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [46]:
print("best param :",grid.best_params_)
print("best Score :",grid.best_score_)
print("best Model :",grid.best_estimator_)

best param : {'estimator__max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 150}
best Score : 0.7135
best Model : AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   learning_rate=0.5, n_estimators=150, random_state=42)


In [47]:
best_model=grid.best_estimator_

In [48]:
best_pred=best_model.predict(x_test)

In [49]:
x_test.head()

,Age,Annual_Income,Loan_Amount,Credit_Score,Employment_Years,Debt_To_Income,Previous_Defaults,Credit_Utilization,Home_Ownership,Employment_Type
1178,33,627563,219912,744,5.5,0.499,0,0.573,1,1
191,52,577841,692069,663,13.8,0.387,0,0.202,1,3
2862,62,1366513,179729,672,7.4,0.260,0,0.476,1,1
4885,31,2562362,549402,659,7.0,0.426,0,0.613,2,1
4054,21,1293468,472692,673,1.1,0.648,0,0.541,3,1


- Age: 28
- Annual income: 450000
- Loan amount: 300000
- Credit score: 580
- Previous defaults: 1

In [56]:
x_train

,Age,Annual_Income,Loan_Amount,Credit_Score,Employment_Years,Debt_To_Income,Previous_Defaults,Credit_Utilization,Home_Ownership,Employment_Type
1529,66,764963,1311553,533,6.7,0.149,0,0.155,1,3
2123,51,830670,383169,746,16.8,0.065,0,0.194,1,2
2235,60,687092,424242,761,4.7,0.198,0,0.176,1,3
1928,30,1052142,795996,715,1.0,0.135,0,0.269,1,1
1707,66,409772,592289,536,9.8,0.330,0,0.511,2,1
...,...,...,...,...,...,...,...,...,...,...
733,59,801100,1484633,605,7.6,0.584,0,0.511,1,1
476,63,322740,262373,604,10.9,0.311,0,0.386,1,1
2177,45,1035191,611475,734,8.8,0.143,0,0.319,3,1
695,33,3444182,895449,814,9.1,0.441,0,0.734,2,1


In [64]:
newdata=pd.DataFrame({
    "Age":[28],
    "Annual_Income":[450000],
    "Loan_Amount":[300000],
    "Credit_Score":[810],
    'Employment_Years':[8.5],
    'Debt_To_Income':[0.15],
    "Previous_Defaults":[0],
    'Credit_Utilization':[0.80],
    'Home_Ownership':[1],
    'Employment_Type':[0],
})

In [65]:
print("Prediction :",best_model.predict(newdata))
print("Probabilty :",best_model.predict_proba(newdata))


Prediction : [0]
Probabilty : [[0.73478145 0.26521855]]
